# Plot the simulated body and its ATFA

### Things we have to import

In [1]:
import numpy as np

In [2]:
import matplotlib.pyplot as plt

In [3]:
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

In [4]:
import time

In [5]:
import pickle

In [6]:
from gravmag import data_structures, utils, aux_vedo, transforms, inverse_distance, convolve, filters, constants, eqlayer
from gravmag.models import rectangular_prism as rp

In [7]:
import vedo

In [8]:
# set the backend for vedo
vedo.settings.default_backend= 'vtk'
#vedo.settings.default_backend= '2d'

### Load the model

In [9]:
model_area = pickle.load(open('model/1_model_area.pickle', 'rb'))
model_shape = pickle.load(open('model/1_model_shape.pickle', 'rb'))

In [10]:
model_dx, model_dy = data_structures.grid_xy_spacing(area=model_area, shape=model_shape)
print('model dx = {:.3f} m | model dy = {:.3f} m'.format(model_dx, model_dy))

model dx = 256.410 m | model dy = 204.082 m


In [11]:
boundary_points = pickle.load(open('model/1_model_boundary_points.pickle', 'rb'))

In [12]:
model = pickle.load(open('model/1_model.pickle', 'rb'))

In [13]:
mag_intensity = pickle.load(open('model/1_model_mag_intensity.pickle', 'rb'))
mag_inc = pickle.load(open('model/1_model_mag_inc.pickle', 'rb'))
mag_dec = pickle.load(open('model/1_model_mag_dec.pickle', 'rb'))

In [14]:
# model properties
print("(xmin, xmax) = ({:.3f}, {:.3f}) m".format(model['x1'].min(), model['x2'].max()))
print("(ymin, ymax) = ({:.3f}, {:.3f}) m".format(model['y1'].min(), model['y2'].max()))
print("(top min, top max) = ({:.3f}, {:.3f}) m".format(model['z1'].min(), model['z1'].max()))
print("(bottom min, bottom max) = ({:.3f}, {:.3f}) m".format(model['z2'].min(), model['z2'].max()))
print("model thicknesses: (min, max) = ({:.3f}, {:.3f}) m".format(
    (model['z2'] - model['z1']).min(), (model['z2'] - model['z1']).max()))
print("number of prisms: {}".format(model['x1'].size))

(xmin, xmax) = (-3600.986, 4900.000) m
(ymin, ymax) = (-3000.333, 5800.167) m
(top min, top max) = (0.000, 300.000) m
(bottom min, bottom max) = (300.001, 600.001) m
model thicknesses: (min, max) = (0.001, 600.001) m
number of prisms: 1241


### Load data

In [15]:
boundary_points = pickle.load(open('model/1_model_boundary_points.pickle', 'rb'))

In [16]:
# append the first point at the end to plot a closed curve
boundary_points = np.vstack([boundary_points, boundary_points[0]])

In [17]:
model = pickle.load(open('model/1_model.pickle', 'rb'))

In [18]:
model_area = pickle.load(open('model/1_model_area.pickle', 'rb'))
model_shape = pickle.load(open('model/1_model_shape.pickle', 'rb'))

In [19]:
mag_intensity = pickle.load(open('model/1_model_mag_intensity.pickle', 'rb'))
mag_inc = pickle.load(open('model/1_model_mag_inc.pickle', 'rb'))
mag_dec = pickle.load(open('model/1_model_mag_dec.pickle', 'rb'))

In [20]:
main_field = pickle.load(open('model/2_main_field.pickle', 'rb'))

In [21]:
print('Main field inc = {}° dec = {}°'.format(main_field['inc'], main_field['dec']))

Main field inc = -30° dec = 10°


In [22]:
simulated_fields = pickle.load(open('model/2_simulated_fields.pickle', 'rb'))

In [23]:
coordinates_grid = pickle.load(open('model/2_coordinates_grid.pickle', 'rb'))

In [24]:
print('Regular grid of coordinates on the horizontal plane {} m'.format(coordinates_grid['z']))

Regular grid of coordinates on the horizontal plane -150 m


### Get preliminary information from loaded data

In [25]:
# Cartesian components of the model magnetization
v_hat = utils.unit_vector(inc=mag_inc[0], dec=mag_dec[0])

In [26]:
# compute a normalized horizontal projection of v_hat to plot
v_hat_horizontal = v_hat[:2]/np.linalg.norm(v_hat[:2])

In [27]:
print('Synthetic source magnetization inc = {}° dec = {}°'.format(mag_inc[0], mag_dec[0]))

Synthetic source magnetization inc = -10.2° dec = -19.8°


In [28]:
# Grid spacing
dx, dy = data_structures.grid_xy_spacing(coordinates_grid['area'], coordinates_grid['shape'])

In [29]:
X, Y = data_structures.grid_xy_to_full_matrices_view(
    x = coordinates_grid['x'],
    y = coordinates_grid['y'],
    shape = coordinates_grid['shape']
)

In [30]:
# Cartesian components of the main field
F_hat = utils.unit_vector(inc=main_field['inc'], dec=main_field['dec'])

In [31]:
# Cartesian components of the dipoles forming the equivalent layers
h_hat = v_hat.copy()

In [32]:
print('Layer magnetization inc = {}° dec = {}°'.format(mag_inc[0], mag_dec[0]))

Layer magnetization inc = -10.2° dec = -19.8°


In [33]:
# compute a normalized horizontal projection of F_hat and h_hat to plot
F_hat_horizontal = F_hat[:2]/np.linalg.norm(F_hat[:2])
h_hat_horizontal = h_hat[:2]/np.linalg.norm(h_hat[:2])

### 3D Plot

In [38]:
model = pickle.load(open('model/1_model.pickle', 'rb'))
# change model for plotting
# model_dz = model['z2'] - model['z1']
# model_cz = 0.5*(model['z2'] + model['z1'])
# model['z1'] = model_cz + 2000. - 3*0.5*model_dz
# model['z2'] = model_cz + 2000. + 3*0.5*model_dz

In [39]:
# create a Vedo model of prisms with colors defined by the total-magnetization intensity
model_vedo = aux_vedo.gravmag2vedo_prisms(
    gravmag_prisms=model,
    scalar_props=mag_intensity,
    cmap='jet',
    vmin=np.min(mag_intensity),
    vmax=np.max(mag_intensity)
)

In [40]:
full_coordinates_grid = data_structures.grid_xy_to_full_flatten(
    grid=coordinates_grid, grid_orientation='xy'
)
ranges_atfa = np.max(np.abs(simulated_fields['atfa']))
tfa_vedo = aux_vedo.surface(
    data=np.vstack([full_coordinates_grid['x'], full_coordinates_grid['y'], full_coordinates_grid['z']-1000]).T,
    scalar_props=simulated_fields['atfa'],
    cmap='seismic', vmin=-ranges_atfa, vmax=ranges_atfa
)

A colorized Vedo surface was created


In [42]:
pl = vedo.Plotter()

pl += tfa_vedo.alpha(0.8)

pl += model_vedo

pl += aux_vedo.custom_axes(coordinates_grid['area']+[-1500, 1500])

# set a model view
pl.show(
    axes=0, 
    zoom=1, 
    interactive=True,
    #viewup=(-0.3591183514135307, -0.23430478783991432, -0.9034020567130234), 
    elevation=-260, roll=1, azimuth=0
)

# save the chosen view
pl.show(axes=0, zoom=0.95, interactive=False)
pl.screenshot('model/body_atfa.png')
pl.interactive()

pl.close()